[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AImotion-Bavaria/kipro_wi/blob/main/lecture2/available_workers_ortools.ipynb)

# Available Workers with OR-Tools

Every **available** worker has to work on at least one task.
This example shows `forall` with a `where` clause, `exists`, and set variables in OR-Tools.

In [ ]:
!pip install -q ortools

## Setup

The drawing code lives in `available_workers_viz.py`. On Colab it is downloaded from the
GitHub repository (locally it is found next to the notebook).

When running locally, `autoreload` re-imports the module automatically, so changes to
`available_workers_viz.py` take effect without restarting the kernel. On Colab it is switched off.

In [ ]:
import os
import urllib.request

if not os.path.exists("available_workers_viz.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/AImotion-Bavaria/kipro_wi/main/lecture2/available_workers_viz.py",
        "available_workers_viz.py")

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

# autoreload only for local development (not needed on Colab)
if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

import matplotlib.pyplot as plt
import available_workers_viz as viz

## Data

```minizinc
int: n_workers = 6;
set of int: Workers = 1..n_workers;
int: n_tasks = 3;
set of int: Tasks = 1..n_tasks;

array[Workers] of bool: available;
available = [false,true,false,true,true,true];
```

In [ ]:
n_workers = 6
Workers = range(1, n_workers + 1)
n_tasks = 3
Tasks = range(1, n_tasks + 1)

# a dict keeps the MiniZinc indices 1..6
available = dict(zip(Workers, [False, True, False, True, True, True]))

## Model

```minizinc
array[Tasks] of var set of Workers: worked_on_by;

constraint forall(w in Workers where available[w]) (
               exists(t in Tasks) (w in worked_on_by[t])
);
```

CP-SAT has no set variables. As usual we use **one Boolean per possible element**:
`worked_on_by[t][w]` is true $\Leftrightarrow$ worker `w` is in the set of task `t`.

* `forall(w in Workers where available[w])` becomes a Python `for` loop with an `if`
* `exists(t in Tasks) (...)` becomes `add_bool_or`: at least one of the Booleans is true

In [ ]:
from ortools.sat.python import cp_model


def build_model():
    model = cp_model.CpModel()

    # array[Tasks] of var set of Workers: worked_on_by;
    worked_on_by = {t: {w: model.new_bool_var(f"{w}_in_worked_on_by[{t}]") for w in Workers}
                    for t in Tasks}

    # forall(w in Workers where available[w]) (exists(t in Tasks) (w in worked_on_by[t]))
    for w in Workers:
        if available[w]:
            model.add_bool_or([worked_on_by[t][w] for t in Tasks])

    return model, worked_on_by


def solve(model, worked_on_by):
    """Solve and return the solution as dict task -> set of workers."""
    solver = cp_model.CpSolver()
    status = solver.solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        print("=====UNSATISFIABLE=====")
        return None
    solution = {t: {w for w in Workers if solver.value(worked_on_by[t][w])} for t in Tasks}
    for t in Tasks:
        print(f"worked_on_by[{t}] = {sorted(solution[t])}")
    return solution

## Solve and visualise

In [ ]:
model, worked_on_by = build_model()
solution = solve(model, worked_on_by)

viz.draw_assignment(solution, available)
plt.show()

## What about workers who are *not* available?

The model only says that available workers must work. It does **not** forbid that
unavailable workers are assigned to a task, too!

To make this visible, we ask the solver to assign **as many workers as possible**.
Unavailable workers are drawn grey, their assignments as red dashed lines.

In [ ]:
model, worked_on_by = build_model()
model.maximize(sum(worked_on_by[t][w] for t in Tasks for w in Workers))
solution = solve(model, worked_on_by)

viz.draw_assignment(solution, available, title="maximal assignment: original model")
plt.show()

## Fix: unavailable workers do not work

```minizinc
constraint forall(w in Workers where not available[w]) (
               forall(t in Tasks) (not (w in worked_on_by[t]))
);
```

In [ ]:
model, worked_on_by = build_model()

for w in Workers:
    if not available[w]:
        for t in Tasks:
            model.add(worked_on_by[t][w] == 0)

model.maximize(sum(worked_on_by[t][w] for t in Tasks for w in Workers))
solution = solve(model, worked_on_by)

viz.draw_assignment(solution, available, title="maximal assignment: with the fix")
plt.show()